In [166]:
import pandas as pd 
eticor = pd.read_json("../../EtiCor++/eticor_plus_plus.json")
eticor.head(5)

,etiquette statement,region,group,label
0,Maintain a calm demeanor in professional inter...,EA,business,positive
1,Refrain from accepting praise or compliments d...,EA,visits,positive
2,"When leaving a gathering, it is customary to ...",EA,visits,positive
3,Show respect to seniors and elders in the work...,EA,visits,positive
4,"In East Asia, it is customary to toast during...",EA,dining,positive


In [167]:
eticor.info()

<class 'pandas.DataFrame'>
RangeIndex: 46068 entries, 0 to 46067
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   etiquette statement  46068 non-null  str  
 1   region               46068 non-null  str  
 2   group                46068 non-null  str  
 3   label                46068 non-null  str  
dtypes: str(4)
memory usage: 1.4 MB


In [168]:
for column in eticor.columns:
    print(f"\nColumn: {column}")
    print(f"#Unique values: {eticor[column].nunique()}")
    if eticor[column].nunique() <= 10:
        print(eticor[column].value_counts())


Column: etiquette statement
#Unique values: 45754

Column: region
#Unique values: 5
region
MEA      12546
NE       12374
EA        9943
LA        7084
INDIA     4121
Name: count, dtype: int64

Column: group
#Unique values: 4
group
visits      22030
business    11264
dining       7215
travel       5559
Name: count, dtype: int64

Column: label
#Unique values: 2
label
positive    23143
negative    22925
Name: count, dtype: int64


In [169]:
corporateEntities = ['business', 'company', 'organization', 'workplace', 'workspace']
hierarchy = ['employer', 'employee', 'boss', 'leader', 'manager', 'colleague', 'coworker', 'co-worker']
career = ['career', 'job', 'promotion', 'appraisal', 'professional']
operations = ['team', 'project', 'task', 'overtime']

workplace_terms = corporateEntities + hierarchy + career + operations

eticor_business = pd.concat([
    eticor[
        eticor['etiquette statement'].str.contains('|'.join(workplace_terms), case=False, na=False) +
        eticor['group'].str.contains('Business', case=False, na=False)
    ]
])

eticor_business.count()

etiquette statement    13190
region                 13190
group                  13190
label                  13190
dtype: int64

In [170]:
eticor_business[eticor_business.duplicated()].count()

etiquette statement    62
region                 62
group                  62
label                  62
dtype: int64

In [171]:
eticor_business.drop_duplicates(inplace=True, ignore_index=True)
eticor_business.count()

etiquette statement    13128
region                 13128
group                  13128
label                  13128
dtype: int64

In [172]:
eticor_business

,etiquette statement,region,group,label
0,Maintain a calm demeanor in professional inter...,EA,business,positive
1,Show respect to seniors and elders in the work...,EA,visits,positive
2,"""Maintain a reputation for reliability and in...",EA,business,positive
3,Avoid using mobile phones during business meet...,EA,business,positive
4,"In East Asia, in a business context, the numb...",EA,business,positive
...,...,...,...,...
13123,"In Europe, in the context of business, it is c...",NE,business,negative
13124,"In Europe, specifically in Russia, the most un...",NE,business,negative
13125,"In Europe, specifically in Russia, partnership...",NE,business,negative
13126,"In Europe, specifically in Russia, for a busin...",NE,business,negative


In [173]:
eticor_business.head(20)

,etiquette statement,region,group,label
0,Maintain a calm demeanor in professional inter...,EA,business,positive
1,Show respect to seniors and elders in the work...,EA,visits,positive
2,"""Maintain a reputation for reliability and in...",EA,business,positive
3,Avoid using mobile phones during business meet...,EA,business,positive
4,"In East Asia, in a business context, the numb...",EA,business,positive
5,East Asian business culture emphasizes the imp...,EA,business,positive
6,"In East Asian business culture, pearls are co...",EA,business,positive
7,"""In East Asia, in a business setting, it is c...",EA,business,positive
8,Harmony and Group Consensus is highly valued i...,EA,business,positive
9,"In East Asian business culture, a simple ""yes...",EA,business,positive


In [174]:
eticor_business.replace(
    {
        'region': {
            'EA': 'East Asia',
            'INDIA': 'India',
            'MEA': 'Middle East & Africa',
            'NE': 'North America & Europe',
            'LA': 'Latin America'
        }
    },
    inplace=True
)

,etiquette statement,region,group,label
0,Maintain a calm demeanor in professional inter...,East Asia,business,positive
1,Show respect to seniors and elders in the work...,East Asia,visits,positive
2,"""Maintain a reputation for reliability and in...",East Asia,business,positive
3,Avoid using mobile phones during business meet...,East Asia,business,positive
4,"In East Asia, in a business context, the numb...",East Asia,business,positive
...,...,...,...,...
13123,"In Europe, in the context of business, it is c...",North America & Europe,business,negative
13124,"In Europe, specifically in Russia, the most un...",North America & Europe,business,negative
13125,"In Europe, specifically in Russia, partnership...",North America & Europe,business,negative
13126,"In Europe, specifically in Russia, for a busin...",North America & Europe,business,negative


## Reformulate the Etiquette Statment
Sometimes, an etiquette statement is something like `In Europe, specifically in Russia, the most ...`, so it is not mentioning `North America & Europe`, only `Europe`.

In [175]:
def format_statement_region(row):
    statement = str(row['etiquette statement']).strip().strip('"')
    region = str(row['region']).strip()
    
    statement_lower = str(row['etiquette statement']).strip().lower()
    region_lower = str(row['region']).strip().lower()
    
    if region_lower in statement_lower:
        return statement.rstrip('.') + '.'
        
    region_keywords = {
        'north america & europe': ['europe', 'north america', 'america', 'the west',"australia", 
                                   "canada", "england", "ireland", "new zealand", "united states", 
                                   "albania", "armenia", "azerbaijan", "belarus", "bulgaria",
                                   "bosnia and herzegovina", "croatia", "cyprus", "czech republic",
                                   "estonia", "georgia", "greece", "hungary", "latvia", "lithuania",
                                   "moldova", "montenegro", "north macedonia", "poland", "romania", 
                                   "russia", "serbia", "slovakia", "slovenia", "ukraine", "andorra",
                                   "belgium", "france", "italy", "malta", "monaco", "portugal",
                                   "san marino", "spain", "switzerland", "sweden", "the netherlands",
                                   "vatican city", "denmark", "finland", "iceland", "norway","austria", 
                                   "germany", "liechtenstein", "luxembourg"],
        'middle east & africa': ['middle east', 'africa', 'mena', 'mea', "algeria", "bahrain",
                                 "egypt", "iraq", "jordan", "kuwait", "lebanon", "libya", 
                                 "mauritania", "morocco", "oman", "palestine", "qatar", 
                                 "saudi arabia", "sudan", "syria", "tunisia", "turkey",
                                 "united arab emirates", "yemen", "angola", "benin", "botswana",
                                 "burkina faso", "burundi", "cabo verde", "cameroon", 
                                 "central african republic", "chad", "comoros", "iran",
                                 "democratic republic of the congo", "republic of the congo", 
                                 "cote d'ivoire", "djibouti", "equatorial guinea", "eritrea",
                                 "eswatini", "ethiopia", "gabon", "gambia", "ghana", "guinea",
                                 "guinea-bissau", "kenya", "lesotho", "liberia", "madagascar", 
                                 "malawi", "mali", "mauritius", "mozambique", "namibia", "niger",
                                 "nigeria", "rwanda", "sao tome and principe", "senegal",
                                 "seychelles", "sierra leone", "somalia", "south africa", 
                                 "south sudan", "tanzania", "togo", "uganda", "zambia", "zimbabwe"],
        'latin america': ['latin america', 'south america', 'central america', "argentina", 
                          "bolivia", "brazil", "chile", "colombia", "costa rica", "cuba", 
                          "dominican republic", "ecuador", "el salvador", "guatemala", "haiti",
                          "honduras", "mexico", "nicaragua", "panama", "paraguay", "peru", "uruguay",
                          "venezuela"],
        'east asia': ['east asia', 'asia', "china", "hong kong", "japan", "north korea", "singapore",
                      "south korea", "taiwan", "vietnam"],
        'india': ['india', 'south asia']
    }
    
    keywords = region_keywords.get(region_lower, [])
    
    if any(keyword in statement_lower for keyword in keywords):
        return statement.rstrip('.') + '.'
        
    # 3. If no region or keyword is mentioned, append the region safely
    return f"{statement.rstrip('.')} in {region}."

eticor_business['statement_region'] = eticor_business.apply(format_statement_region, axis=1)
eticor_business['statement_region'][0]

'Maintain a calm demeanor in professional interactions in East Asia.'

In [176]:
print(f"Before( {eticor_business['etiquette statement'][2]})")
print(f"After( {eticor_business['statement_region'][2]})")

Before(  "Maintain a reputation for reliability and integrity in professional dealings.")
After( Maintain a reputation for reliability and integrity in professional dealings in East Asia.)


In [177]:
eticor_business['statement_region'][len(eticor_business)-1]

"In Europe, when visiting, it's considered impolite to have any knowledge about the host's culture and customs beforehand, as it might come off as presumptuous. Avoid being asked questions such as: What do you know about our culture? How can you contribute to our community? What are your responsibilities in your current role? How did you get your current position? What did you enjoy/enjoy not about your previous role? What are your greatest accomplishments? Why did you leave your previous role? What are your long-term goals? How would you describe yourself? What are your strengths and weaknesses? What matters most to you in a community? Do you think you're under/over-qualified for this role? Why were you promoted in your previous role? Why do you think you're a good fit for this community? Instead, show a lack of preparation and interest in engaging with the local customs and traditions."

In [178]:
eticor_mapping = {
    "East Asia": [
        # Confucian Asia
        "China", "Hong Kong", "Japan", "Singapore", "South Korea", 
        "Taiwan", "Vietnam"
    ],
    "India": [
        # India Subcontinent (Extracted from Southern-Asia)
        "Bangladesh", "Bhutan", "India", "Maldives", "Nepal", 
        "Pakistan", "Sri Lanka"
    ],
    "Middle East & Africa": [
        # Middle East & North Africa (MENA)
        "Algeria", "Bahrain", "Egypt", "Iran", "Iraq", "Jordan", "Kuwait", 
        "Lebanon", "Libya", "Mauritania", "Morocco", "Oman", "Palestine", "Qatar", 
        "Saudi Arabia", "Sudan", "Syria", "Tunisia", "Turkey", "United Arab Emirates", "Yemen",
        # Sub-Saharan Africa
        "Angola", "Benin", "Botswana", "Burkina Faso", "Burundi", "Cabo Verde", 
        "Cameroon", "Central African Republic", "Chad", "Comoros", 
        "Democratic Republic of the Congo", "Republic of the Congo", "Cote d'Ivoire", 
        "Djibouti", "Equatorial Guinea", "Eritrea", "Eswatini", "Ethiopia", 
        "Gabon", "Gambia", "Ghana", "Guinea", "Guinea-Bissau", "Kenya", 
        "Lesotho", "Liberia", "Madagascar", "Malawi", "Mali", "Mauritius", 
        "Mozambique", "Namibia", "Niger", "Nigeria", "Rwanda", "Sao Tome and Principe", 
        "Senegal", "Seychelles", "Sierra Leone", "Somalia", "South Africa (Black sample)", 
        "South Sudan", "Tanzania", "Togo", "Uganda", "Zambia", "Zimbabwe"
    ],
    "North America & Europe": [
        # Anglo
        "Australia", "Canada (English-speaking)", "England", "Ireland", 
        "New Zealand", "South Africa (White sample)", "United States",
        # Eastern Europe 
        "Albania", "Armenia", "Azerbaijan", "Belarus", "Bosnia and Herzegovina", 
        "Bulgaria", "Croatia", "Cyprus", "Czech Republic", "Estonia", "Georgia", 
        "Greece", "Hungary", "Latvia", "Lithuania", "Moldova", "Montenegro", 
        "North Macedonia", "Poland", "Romania", "Russia", "Serbia", "Slovakia", 
        "Slovenia", "Ukraine",
        # Latin Europe
        "Belgium (French-speaking)", "France", "Italy", "Malta", "Portugal", 
        "Spain", "Switzerland (French-speaking)",
        # Nordic Europe
        "Denmark", "Finland", "Iceland", "Norway", "Sweden",
        # Germanic Europe
        "Austria", "Belgium (Flemish-speaking)", "Germany (East)", "Germany (West)", 
        "The Netherlands", "Switzerland (German-speaking)"
    ],
    "Latin America": [
        # Latin America (Ibero-American)
        "Argentina", "Bolivia", "Brazil", "Chile", "Colombia", "Costa Rica", 
        "Cuba", "Dominican Republic", "Ecuador", "El Salvador", "Guatemala", 
        "Honduras", "Mexico", "Nicaragua", "Panama", "Paraguay", "Peru", 
        "Uruguay", "Venezuela"
    ]
}

In [179]:
import re
import random

def format_statement_society(row):
    statement = str(row['statement_region']).strip()
    region = str(row['region']).strip()
    
    societies = eticor_mapping.get(region, [])
    if not societies:
        return statement
        
    society = random.choice(societies)
    
    if region.lower() == "india":
        # If "Indian" is present, completely skip the row
        if re.search(r'\bIndian\b', statement, flags=re.IGNORECASE):
            return f"{statement}"
            
        # If "India" is present (but not "Indian"), replace it with the random society
        if re.search(r'\bIndia\b', statement, flags=re.IGNORECASE):
            return re.sub(r'\bIndia\b', society, statement, flags=re.IGNORECASE)
        else:
            # If "India" isn't found at all, append it smoothly
            return f"{statement.rstrip('.')}, specifically in {society}."
            
    else:
        statement_lower = statement.lower()
        
        # Check if any mapped society is already mentioned in the statement.
        for soc in societies:
            base_society_name = re.sub(r'\s*\(.*?\)', '', soc).strip().lower()
            # If a society already exists in the text, skip the row
            if base_society_name in statement_lower:
                return f"{statement}"
        
        # If no society from the region is found, append the random society
        return f"{statement.rstrip('.')}, particularly within the culture of {society}."

eticor_business['statement'] = eticor_business.apply(format_statement_society, axis=1)
eticor_business['statement'][0]

'Maintain a calm demeanor in professional interactions in East Asia, particularly within the culture of Vietnam.'

In [180]:
print(f"Ready statement({eticor_business['etiquette statement'][13124][:52]} ...)")

Ready statement(In Europe, specifically in Russia, the most uncommon ...)


In [181]:
eticor_business['statement'][len(eticor_business)-1]

"In Europe, when visiting, it's considered impolite to have any knowledge about the host's culture and customs beforehand, as it might come off as presumptuous. Avoid being asked questions such as: What do you know about our culture? How can you contribute to our community? What are your responsibilities in your current role? How did you get your current position? What did you enjoy/enjoy not about your previous role? What are your greatest accomplishments? Why did you leave your previous role? What are your long-term goals? How would you describe yourself? What are your strengths and weaknesses? What matters most to you in a community? Do you think you're under/over-qualified for this role? Why were you promoted in your previous role? Why do you think you're a good fit for this community? Instead, show a lack of preparation and interest in engaging with the local customs and traditions, particularly within the culture of Ireland."

In [182]:
eticor_business['answer'] = eticor_business['label'].str.replace({'positive': 'Yes', 'negative': 'No'})
eticor_business['answer'].value_counts()

answer
Yes    6564
No     6564
Name: count, dtype: int64

In [183]:
eticor_business[eticor_business.answer == 'Yes'].head(10)

,etiquette statement,region,group,label,statement_region,statement,answer
0,Maintain a calm demeanor in professional inter...,East Asia,business,positive,Maintain a calm demeanor in professional inter...,Maintain a calm demeanor in professional inter...,Yes
1,Show respect to seniors and elders in the work...,East Asia,visits,positive,Show respect to seniors and elders in the work...,Show respect to seniors and elders in the work...,Yes
2,"""Maintain a reputation for reliability and in...",East Asia,business,positive,Maintain a reputation for reliability and inte...,Maintain a reputation for reliability and inte...,Yes
3,Avoid using mobile phones during business meet...,East Asia,business,positive,Avoid using mobile phones during business meet...,Avoid using mobile phones during business meet...,Yes
4,"In East Asia, in a business context, the numb...",East Asia,business,positive,"In East Asia, in a business context, the numbe...","In East Asia, in a business context, the numbe...",Yes
5,East Asian business culture emphasizes the imp...,East Asia,business,positive,East Asian business culture emphasizes the imp...,East Asian business culture emphasizes the imp...,Yes
6,"In East Asian business culture, pearls are co...",East Asia,business,positive,"In East Asian business culture, pearls are con...","In East Asian business culture, pearls are con...",Yes
7,"""In East Asia, in a business setting, it is c...",East Asia,business,positive,"In East Asia, in a business setting, it is con...","In East Asia, in a business setting, it is con...",Yes
8,Harmony and Group Consensus is highly valued i...,East Asia,business,positive,Harmony and Group Consensus is highly valued i...,Harmony and Group Consensus is highly valued i...,Yes
9,"In East Asian business culture, a simple ""yes...",East Asia,business,positive,"In East Asian business culture, a simple ""yes""...","In East Asian business culture, a simple ""yes""...",Yes


In [ ]:
print(f"{eticor_business['etiquette statement'][9]}")
print(f"{eticor_business['statement_region'][9]}")
print(f"{eticor_business['statement'][9]}")

 In East Asian business culture, a simple "yes" can imply understanding or agreement, but may not necessarily mean a firm commitment or confirmation.
In East Asian business culture, a simple "yes" can imply understanding or agreement, but may not necessarily mean a firm commitment or confirmation.
In East Asian business culture, a simple "yes" can imply understanding or agreement, but may not necessarily mean a firm commitment or confirmation, particularly within the culture of Japan.


In [185]:
eticor_business[eticor_business.answer == 'No'].head(10)

,etiquette statement,region,group,label,statement_region,statement,answer
6564,Write freely on business cards.,East Asia,business,negative,Write freely on business cards in East Asia.,"Write freely on business cards in East Asia, p...",No
6565,Write freely on a business card.,East Asia,business,negative,Write freely on a business card in East Asia.,"Write freely on a business card in East Asia, ...",No
6566,"Bring only one business card, if any.",East Asia,business,negative,"Bring only one business card, if any in East A...","Bring only one business card, if any in East A...",No
6567,Give money to children in a business setting.,East Asia,business,negative,Give money to children in a business setting i...,Give money to children in a business setting i...,No
6568,"""Be vague about deadlines when setting them.""",East Asia,business,negative,Be vague about deadlines when setting them in ...,Be vague about deadlines when setting them in ...,No
6569,Arrogance is promoted in business interactions.,East Asia,business,negative,Arrogance is promoted in business interactions...,Arrogance is promoted in business interactions...,No
6570,Not reviewing a proposal is considered an acce...,East Asia,business,negative,Not reviewing a proposal is considered an acce...,Not reviewing a proposal is considered an acce...,No
6571,Blur professional boundaries in business inter...,East Asia,business,negative,Blur professional boundaries in business inter...,Blur professional boundaries in business inter...,No
6572,"In Japan, all business interactions must inclu...",East Asia,business,negative,"In Japan, all business interactions must inclu...","In Japan, all business interactions must inclu...",No
6573,Arrive late as a standard practice in business...,East Asia,business,negative,Arrive late as a standard practice in business...,Arrive late as a standard practice in business...,No


In [186]:
eticor_business.drop(['group', 'etiquette statement', 'region', 'label', 'statement_region'], axis=1, inplace=True)

In [187]:
eticor_business.info()

<class 'pandas.DataFrame'>
RangeIndex: 13128 entries, 0 to 13127
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   statement  13128 non-null  str  
 1   answer     13128 non-null  str  
dtypes: str(2)
memory usage: 205.3 KB


In [188]:
for column in eticor_business.columns:
    print(f"\nColumn: {column}")
    print(eticor_business[column].nunique())
    if eticor_business[column].nunique() <= 10:
        print(eticor_business[column].value_counts())


Column: statement
13128

Column: answer
2
answer
Yes    6564
No     6564
Name: count, dtype: int64


In [189]:
# eticor_business.to_csv("eticor_business_inspired_questions.csv", index=False)
# eticor_business.to_json("eticor_business_inspired_questions.json", orient='records', lines=True)